# Diabetes 130-US Hospitals (1999-2008) — Exploratory Data Analysis

General-overview EDA of the [Diabetes 130-US hospitals dataset](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008) (also mirrored on Kaggle). The dataset contains ~10 years of clinical care records for diabetic patients across 130 US hospitals, with the goal of studying hospital readmission.

**Files used**
- `diabetic_data.csv` — main encounter-level table (101,766 rows x 50 columns)
- `IDS_mapping.csv` — lookup tables that decode the numeric ID columns (`admission_type_id`, `discharge_disposition_id`, `admission_source_id`)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 60)


## 1. Load data

Works both on Kaggle (reading from `/kaggle/input/...`) and locally (reading from the sibling `diabetes+130-us+hospitals+for+years+1999-2008/` folder). If you add this dataset to a Kaggle notebook, update `kaggle_dir` to match the exact input folder name shown in the sidebar.

In [ ]:
import os

kaggle_dir = "/kaggle/input/diabetes-130-us-hospitals-for-years-1999-2008"
local_dir = "diabetes+130-us+hospitals+for+years+1999-2008"

data_dir = kaggle_dir if os.path.exists(kaggle_dir) else local_dir

df = pd.read_csv(os.path.join(data_dir, "diabetic_data.csv"))
ids_map = pd.read_csv(os.path.join(data_dir, "IDS_mapping.csv"))

print(f"Loaded from: {data_dir}")
df.shape


## 2. First look

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="number").T

## 3. Missing values

This dataset encodes missing values as the string `"?"` rather than `NaN`, so a plain `isna().sum()` will under-report missingness. We replace `"?"` with `NaN` first.

In [ ]:
df_clean = df.replace("?", np.nan)

missing = df_clean.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
missing_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=missing_summary["missing_pct"], y=missing_summary.index, ax=ax, color="steelblue")
ax.set_xlabel("% missing")
ax.set_title("Missing values by column")
plt.tight_layout()
plt.show()


## 4. Duplicates & patient-level structure

Each row is an *encounter* (hospital visit), not a unique patient — the same `patient_nbr` can appear multiple times. This matters for readmission analysis and for correct train/test splitting later.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Unique encounters:", df["encounter_id"].nunique())
print("Unique patients:", df["patient_nbr"].nunique())
print("Encounters per patient (top 5):")
df["patient_nbr"].value_counts().head()


## 5. Target variable: `readmitted`

In [ ]:
order = ["NO", ">30", "<30"]
counts = df["readmitted"].value_counts().reindex(order)

fig, ax = plt.subplots()
sns.barplot(x=counts.index, y=counts.values, order=order, palette="viridis", ax=ax)
ax.set_ylabel("Number of encounters")
ax.set_title("Readmission status distribution")
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center")
plt.tight_layout()
plt.show()


## 6. Numeric feature distributions

Core utilization/severity measures: length of stay, lab procedures, medications, prior visits.

In [ ]:
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, ax in zip(numeric_cols, axes.ravel()):
    sns.histplot(df[col], bins=30, ax=ax, color="teal")
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
df[numeric_cols].describe().T

## 7. Key categorical features

`age` is already binned into 10-year buckets; `race`, `gender`, `A1Cresult`, `max_glu_serum`, `change`, and `diabetesMed` are also of interest.

In [ ]:
age_order = sorted(df["age"].unique(), key=lambda x: int(x.strip("[)").split("-")[0]))

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="age", order=age_order, color="darkorange", ax=ax)
ax.set_title("Age distribution")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, y="race", order=df["race"].value_counts().index, ax=axes[0, 0], color="slateblue")
axes[0, 0].set_title("Race")

sns.countplot(data=df, x="gender", ax=axes[0, 1], color="mediumseagreen")
axes[0, 1].set_title("Gender")

sns.countplot(data=df, x="A1Cresult", order=["None", "Norm", ">7", ">8"], ax=axes[1, 0], color="salmon")
axes[1, 0].set_title("A1C test result")

sns.countplot(data=df, x="max_glu_serum", order=["None", "Norm", ">200", ">300"], ax=axes[1, 1], color="goldenrod")
axes[1, 1].set_title("Max glucose serum test result")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x="change", ax=axes[0], color="cornflowerblue")
axes[0].set_title("Medication changed?")
sns.countplot(data=df, x="diabetesMed", ax=axes[1], color="indianred")
axes[1].set_title("On diabetes medication?")
plt.tight_layout()
plt.show()


## 8. Correlation between numeric features

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix — numeric features")
plt.tight_layout()
plt.show()


## 9. Numeric features vs. readmission

Quick look at whether utilization measures differ by readmission status.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for col, ax in zip(numeric_cols, axes.ravel()):
    sns.boxplot(data=df, x="readmitted", y=col, order=order, ax=ax, palette="Set2")
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 10. Age vs. readmission

In [ ]:
age_readmit = pd.crosstab(df["age"], df["readmitted"], normalize="index").reindex(age_order) * 100
age_readmit = age_readmit[order]

age_readmit.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="viridis")
plt.ylabel("% of encounters")
plt.title("Readmission status by age group")
plt.xticks(rotation=45)
plt.legend(title="readmitted")
plt.tight_layout()
plt.show()


## 11. Summary of initial findings

*(Fill in after reviewing the plots above — e.g. class balance of `readmitted`, which columns need imputation/dropping, notable skew in utilization features, any age/race patterns worth modeling.)*
